# Demo 5: numerical LP vs photon-counting matrix criteria

We classify the same two probability-plane examples as in `demo_2_analytical_vs_numerical.ipynb` using
(a) the numerical LP solver, (b) the analytical K criterion, (c) the photon-counting matrix criterion `analytical_c`,
and (d) the analytical majorization criterion M.

Here CM means the ideal photoelectric photon-counting matrix criterion from
Krishnaswamy et al. arXiv:2603.06237v1, Sec. II,
not the multiplexed click-counting matrix.


In [ ]:
# Make the project importable when running from the notebooks/ directory.
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.config import apply_thread_limits
apply_thread_limits()

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from src.comparison.scan_2d import load_scan_result, run2D
from src.config import SCAN


In [ ]:
def classification_map(data):
    """0 classical, 1 CM-only, 2 LP-only, 3 both, NaN non-physical."""
    num = data['numerical']['nonclass']
    cm = data['counting_matrix']['nonclass']
    valid = np.isfinite(num) & np.isfinite(cm)
    classes = np.full(num.shape, np.nan, dtype=float)
    classes[valid] = 0.0
    classes[valid & (cm > 0.5)] = 1.0
    classes[valid & (num > 0.5)] = 2.0
    classes[valid & (num > 0.5) & (cm > 0.5)] = 3.0
    return classes


def analytical_classification_map(data):
    """0 classical, 1 analytical-only, 2 LP-only, 3 both, NaN non-physical."""
    num = data['numerical']['nonclass']
    ana = data['analytical']['nonclass']
    valid = np.isfinite(num) & np.isfinite(ana)
    classes = np.full(num.shape, np.nan, dtype=float)
    classes[valid] = 0.0
    classes[valid & (ana > 0.5)] = 1.0
    classes[valid & (num > 0.5)] = 2.0
    classes[valid & (num > 0.5) & (ana > 0.5)] = 3.0
    return classes


def plot_all_boundaries(data, xlabel, ylabel, title):
    plt.figure(figsize=(6, 5))
    for b in data['numerical']['boundaries']:
        plt.plot(b[:, 0], b[:, 1], lw=6, color='black', label='LP')
    for b in data['ana_k']['boundaries']:
        plt.plot(b[:, 0], b[:, 1], '-.', lw=4, color='red', label='analytical K')
    for b in data['counting_matrix']['boundaries']:
        plt.plot(b[:, 0], b[:, 1], '--', lw=4, color='green', label='analytical CM')
    for b in data['ana_m']['boundaries']:
        plt.plot(b[:, 0], b[:, 1], ':', lw=4, color='blue', label='analytical M')

    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title); plt.tight_layout(); plt.show()


def plot_analytical_vs_lp_map(data, xlabel, ylabel, title):
    classes = analytical_classification_map(data)
    plt.figure(figsize=(6, 5))
    plt.pcolormesh(data['x'], data['y'],
                   np.ma.masked_invalid(classes),
                   shading='auto', cmap=map_cmap())
    plt.colorbar(label='0 classical / 1 all analytical / 2 LP-only / 3 both')
    for b in data['numerical']['boundaries']:
        plt.plot(b[:, 0], b[:, 1], lw=2, color='red')
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title); plt.tight_layout(); plt.show()


def map_cmap():
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad('white')
    return cmap


## Criterion

The LP solver tests compatibility with mixtures of Poisson distributions. The analytical CM criterion implements the ideal photoelectric counting matrix from Krishnaswamy et al. arXiv:2603.06237v1 and tests positive semidefiniteness of all admissible photon-counting matrices

$$
C^{(I)} = \left[(k+l)! P_{k+l}\right]_{k,l\in I},
$$

where $I$ is the row/column index set. It is either integer-valued or half-integer-valued, and only entries with available probabilities $P_{k+l}$ are used. A negative minimum eigenvalue of any admissible matrix certifies nonclassicality.

The numerical LP criterion is stronger for finite probability data, so K, CM, and M detections should lie inside the LP nonclassical region up to numerical tolerance. The boundary plots compare K, CM, and M directly.


In [ ]:
fi = '../results/demo_5_cm_probability_plane_01.npz'
indx = np.array([0, 1, 2, 3], dtype=np.int16)
print('scan decision threshold bias =', SCAN.bias)

data = run2D(
    N0=120, N1=120,
    indx=indx,
    dPA=1e-10, uPA=0.9, dPB=1e-10, uPB=0.3,
    Prest=np.array([0.05, 0.05]),
    fi=fi,
    use_ana_k=True,
    use_ana_m=True,
    use_cm=True,
)
print('grid:', data['numerical']['nonclass'].shape,
      '| LP curves:', len(data['numerical']['boundaries']),
      '| K curves:', len(data['ana_k']['boundaries']),
      '| CM curves:', len(data['counting_matrix']['boundaries']),
      '| M curves:', len(data['ana_m']['boundaries']),
      '| CM fraction:', data['counting_matrix']['ncaf'])


## Overlay the decision boundaries

The thick black curve is the LP boundary. The red dash-dotted curve is the analytical K boundary, the green dashed curve is the photon-counting matrix boundary, and the blue dotted curve is the analytical M boundary.


In [ ]:
plt.figure(figsize=(6, 5))
for b in data['numerical']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], lw=6, color='black', label='LP')
for b in data['ana_k']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], '-.', lw=4, color='red', label='analytical K')
for b in data['counting_matrix']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], '--', lw=4, color='green', label='analytical CM')
for b in data['ana_m']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], ':', lw=4, color='blue', label='analytical M')

handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())
plt.xlabel(r'$P_0$'); plt.ylabel(r'$P_1$')
plt.title('Nonclassicality boundaries'); plt.tight_layout(); plt.show()


In [ ]:
classes = classification_map(data)
plt.figure(figsize=(6, 5))
plt.pcolormesh(data['x'], data['y'],
               np.ma.masked_invalid(classes),
               shading='auto', cmap=map_cmap())
plt.colorbar(label='0 classical / 1 CM-only / 2 LP-only / 3 both')
for b in data['numerical']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], lw=2, color='red')
plt.xlabel(r'$P_0$'); plt.ylabel(r'$P_1$')
plt.title('Sample nonclassicality map'); plt.tight_layout(); plt.show()


### Second example for $P_1$ and $P_3$ plane and fixed $P_2$, $P_5$, and $P_6$


In [ ]:
fi = '../results/demo_5_cm_probability_plane_13.npz'
indx = np.array([1, 3, 2, 5, 6], dtype=np.int16)

data = run2D(
    N0=200, N1=200,
    indx=indx,
    dPA=1e-10, uPA=0.3, dPB=1e-10, uPB=0.04,
    Prest=np.array([0.05, 0.001, 0.001]),
    fi=fi,
    use_ana_k=True,
    use_ana_m=True,
    use_cm=True,
)
print('grid:', data['numerical']['nonclass'].shape,
      '| LP curves:', len(data['numerical']['boundaries']),
      '| K curves:', len(data['ana_k']['boundaries']),
      '| CM curves:', len(data['counting_matrix']['boundaries']),
      '| M curves:', len(data['ana_m']['boundaries']),
      '| CM fraction:', data['counting_matrix']['ncaf'])


In [ ]:
plt.figure(figsize=(6, 5))
for b in data['numerical']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], lw=6, color='black', label='LP')
for b in data['ana_k']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], '-.', lw=4, color='red', label='analytical K')
for b in data['counting_matrix']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], '--', lw=4, color='green', label='analytical CM')
for b in data['ana_m']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], ':', lw=4, color='blue', label='analytical M')

handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())
plt.xlabel(r'$P_1$'); plt.ylabel(r'$P_3$')
plt.title(r'$P_2=0.05,\, P_5=0.001,\,P_6=0.001$'); plt.tight_layout(); plt.show()


In [ ]:
classes = classification_map(data)
plt.figure(figsize=(6, 5))
plt.pcolormesh(data['x'], data['y'],
               np.ma.masked_invalid(classes),
               shading='auto', cmap=map_cmap())
plt.colorbar(label='0 classical / 1 CM-only / 2 LP-only / 3 both')
for b in data['numerical']['boundaries']:
    plt.plot(b[:, 0], b[:, 1], lw=2, color='red')
plt.xlabel(r'$P_1$'); plt.ylabel(r'$P_3$')
plt.title(r'$P_2=0.05,\, P_5=0.001,\,P_6=0.001$'); plt.tight_layout(); plt.show()


## Reload saved scans

The saved `.npz` archives contain the LP result, all enabled analytical criteria, and their extracted boundaries. Reloading them avoids recomputing the scans when only the plots are needed. In the maps below, class `1` is the union of all analytical detections from K, M, and CM.


In [ ]:
data_saved_01 = load_scan_result('../results/demo_5_cm_probability_plane_01.npz')
print('loaded analytical criteria:', data_saved_01['analytical']['criteria'])

plot_all_boundaries(
    data_saved_01,
    xlabel=r'$P_0$', ylabel=r'$P_1$',
    title='Saved scan boundaries: all criteria',
)
plot_analytical_vs_lp_map(
    data_saved_01,
    xlabel=r'$P_0$', ylabel=r'$P_1$',
    title='Saved scan map: all analytical criteria vs LP',
)


In [ ]:
data_saved_13 = load_scan_result('../results/demo_5_cm_probability_plane_13.npz')
print('loaded analytical criteria:', data_saved_13['analytical']['criteria'])

plot_all_boundaries(
    data_saved_13,
    xlabel=r'$P_1$', ylabel=r'$P_3$',
    title=r'Saved scan boundaries: $P_2=0.05,\, P_5=0.001,\,P_6=0.001$',
)
plot_analytical_vs_lp_map(
    data_saved_13,
    xlabel=r'$P_1$', ylabel=r'$P_3$',
    title=r'Saved scan map: $P_2=0.05,\, P_5=0.001,\,P_6=0.001$',
)


## Further examples

Additional notebooks demonstrate:

- the original K/M analytical comparison,
- nonclassicality maps in the Jaynes--Cummings model,
- probabilistic boson addition,
- high-precision LP settings,
- CSV import/export,
- and diagnostic tools for probability-space scans.

See:

```text
notebooks/demo_2_analytical_vs_numerical.ipynb
notebooks/demo_3_jaynes_cummings.ipynb
notebooks/demo_4_probabilistic_boson_addition.ipynb
```
